In [ ]:
%pip install -r "requirements.txt"

from did_functions import *
import pandas as pd
import kagglehub
import geopandas as gpd
import datetime as dt
import sqlite3
import os

### Hardcoded Values
- NCIC Codes are taken from data since they overlap with Vision Zero data for the period of interest

In [56]:
## Hardcoded Values depending on analysis

# Vision Zero city codes
codes_to_drop = ['1942', '3711', '4313', '0105']

# column and date indicators
date_col = "collision_date"
start_date = '2010-01-01'
end_date = '2016-12-31'
collision_cols = ['case_id', date_col, 'county_city_location', 'collision_severity', 'county_location', 'population', 'weather_1', 'primary_collision_factor', 'pcf_violation_category', 'hit_and_run', 'type_of_collision', 'road_condition_1', 'road_surface', 'lighting', 'alcohol_involved', 'latitude', 'longitude']

# data paths
crash_path = "alexgude/california-traffic-collision-data-from-switrs"
ncic_path = r"../data/raw_data/NCIC Code Jurisdiction List_04242023 - Sheet1.csv"
final_data_path = r"../data/clean_data/California_Collisions_Clean.csv"

Load data from the online source

In [57]:
# Download latest data
data = kagglehub.dataset_download(crash_path)
db_file_path = os.path.join(data, 'switrs.sqlite')
ncic = load_data(ncic_path)

NCIC Code Jurisdiction List_04242023 - Sheet1.csv converted to dataframe.


## Data Cleaning

### Grab data for synthetic control

In [ ]:
# establish sql connection to database path
conn = sqlite3.connect(db_file_path)
# use sql query to pull data
query = f"SELECT {', '.join(collision_cols)} FROM collisions WHERE {date_col} >= '{start_date}' AND {date_col} <= '{end_date}'"
collisions = pd.read_sql_query(query, conn)

,case_id,collision_date,county_city_location,collision_severity,county_location,population,weather_1,primary_collision_factor,pcf_violation_category,hit_and_run,type_of_collision,road_condition_1,road_surface,lighting,alcohol_involved,latitude,longitude
0,4391974,2010-01-06,1942,property damage only,los angeles,>250000,clear,other than driver,other than driver (or pedestrian),not hit and run,hit object,normal,dry,daylight,NaN,NaN,NaN
1,4392005,2010-01-19,3300,fatal,riverside,unincorporated,cloudy,vehicle code violation,improper turning,not hit and run,hit object,normal,dry,dark with street lights,1.0,34.01847,-117.50958
2,4392006,2010-01-13,1935,fatal,los angeles,50000 to 100000,raining,vehicle code violation,traffic signals and signs,not hit and run,broadside,normal,wet,dark with street lights,NaN,NaN,NaN
3,4392007,2010-01-15,1942,fatal,los angeles,>250000,clear,vehicle code violation,pedestrian violation,felony,pedestrian,normal,dry,dark with street lights,1.0,NaN,NaN
4,4392008,2010-01-03,1942,fatal,los angeles,>250000,cloudy,vehicle code violation,traffic signals and signs,not hit and run,broadside,normal,dry,daylight,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2929208,8066678,2016-06-12,3022,property damage only,orange,50000 to 100000,clear,other improper driving,other improper driving,misdemeanor,rear end,normal,dry,daylight,NaN,33.43440,-117.47220
2929209,8071228,2016-05-11,4109,property damage only,san mateo,25000 to 50000,clear,vehicle code violation,speeding,misdemeanor,sideswipe,normal,dry,daylight,NaN,37.44952,-122.18463
2929210,8112338,2016-08-15,1801,pain,lassen,10000 to 25000,clear,vehicle code violation,speeding,not hit and run,rear end,normal,dry,daylight,NaN,NaN,NaN
2929211,8121975,2016-08-18,0790,pain,contra costa,25000 to 50000,clear,vehicle code violation,other hazardous violation,not hit and run,rear end,normal,dry,daylight,NaN,NaN,NaN


### Organize Cities by NCIC Codes

In [69]:
# combine data sets
collisions_coded = pd.merge(collisions, ncic, how='left', left_on='county_city_location', right_on='Code')
# drop vision zero codes from synthetic data
collisions_ca = collisions_coded[~collisions_coded['Code'].isin(codes_to_drop)]

### Label treatment by city and time


In [70]:
collisions_ca[date_col] = pd.to_datetime(collisions_ca[date_col])
collisions_ca['year_month'] = collisions_ca[date_col].dt.to_period('M')

Remove unnecessary entries from agency column

In [71]:
# Get rid of Sheriff's Departments in agencies
collisions_ca = collisions_ca[~collisions_ca['Agency'].str.contains('Sheriff|Police', na=False)]

In [72]:
# Find top 18 cities plus San Francisco in terms of crashes by Agency Name
top_cities = collisions_ca.groupby('Agency')['case_id'].count().sort_values(ascending = False)
top_cities = top_cities.head(19)

In [73]:
# save final dataset
collisions_ca.to_csv(final_data_path)